In [1]:
import pandas as pd

In [2]:
pd.set_option('display.max_row', 1000)
df = pd.read_csv('Data\Train.csv')
df.head()

,ID,text,label
0,SUAVK39Z,I feel that it was better I dieAm happy,Depression
1,9JDAGUV3,Why do I get hallucinations?,Drugs
2,419WR1LQ,I am stresseed due to lack of financial suppor...,Depression
3,6UY7DX6Q,Why is life important?,Suicide
4,FYC0FTFB,How could I be helped to go through the depres...,Depression


In [4]:
train = df.iloc[:, :-1]
test = pd.read_csv('Data/Test.csv')
target = df.iloc[:, -1]

In [5]:
print(train.shape)
print(test.shape)
print(target.shape)

(616, 2)
(309, 2)
(616,)


In [6]:
text = pd.concat([train, test], ignore_index=True)
text.shape

(925, 2)

In [7]:
import re
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Yaçin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

In [9]:
corpus = []
for i in range(0, 925):
    feeling = re.sub('[^a-zA-Z]', ' ', str(text['text'][i]))
    feeling = feeling.lower()
    feeling = feeling.split()
    ps = PorterStemmer()
    feeling = [ps.stem(word) for word in feeling if not word in set(stopwords.words('english'))]
    feeling = ' '.join(feeling)
    corpus.append(feeling)

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=500)
result = cv.fit_transform(corpus).toarray()

In [11]:
result.shape

(925, 500)

In [12]:
dataframe = pd.DataFrame(result)
dataframe.tail()

,0,1,2,3,4,5,6,7,8,9,...,490,491,492,493,494,495,496,497,498,499
920,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
921,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
922,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
923,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
924,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
training = df.iloc[:,0].to_frame()
training = training.join(dataframe.iloc[:616,:])
training = training.join(target)
training.head()

,ID,0,1,2,3,4,5,6,7,8,...,491,492,493,494,495,496,497,498,499,label
0,SUAVK39Z,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Depression
1,9JDAGUV3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Drugs
2,419WR1LQ,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Depression
3,6UY7DX6Q,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Suicide
4,FYC0FTFB,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Depression


In [14]:
training.shape

(616, 502)

In [19]:
testing = test.iloc[:,0].to_frame()
_test = dataframe.iloc[616:,:].reset_index()
_test.drop(columns = 'index', inplace = True)
testing = testing.join(_test)
testing.head()

,ID,0,1,2,3,4,5,6,7,8,...,490,491,492,493,494,495,496,497,498,499
0,02V56KMO,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,03BMGTOK,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,03LZVFM6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0EPULUM5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0GM4C5GD,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [20]:
X = training.iloc[:,1:500].values
y = training.iloc[:,-1].values

In [21]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.15, random_state = 0)

In [22]:
from sklearn.svm import SVC
classifier = SVC(probability=True)
classifier.fit(X_train, y_train)

C:\ProgramData\Anaconda3\lib\site-packages\sklearn\svm\base.py:193: FutureWarning: The default value of gamma will change from 'auto' to 'scale' in version 0.22 to account better for unscaled features. Set gamma explicitly to 'auto' or 'scale' to avoid this warning.
  "avoid this warning.", FutureWarning)


SVC(C=1.0, cache_size=200, class_weight=None, coef0=0.0,
    decision_function_shape='ovr', degree=3, gamma='auto_deprecated',
    kernel='rbf', max_iter=-1, probability=True, random_state=None,
    shrinking=True, tol=0.001, verbose=False)

In [23]:
import numpy as np
y_pred = classifier.predict_proba(X_test)
np.around(y_pred, decimals=2)

array([[0.02, 0.91, 0.03, 0.05],
       [0.02, 0.93, 0.01, 0.04],
       [0.05, 0.84, 0.04, 0.08],
       [0.06, 0.8 , 0.05, 0.1 ],
       [0.06, 0.74, 0.08, 0.12],
       [0.1 , 0.07, 0.55, 0.28],
       [0.03, 0.9 , 0.02, 0.05],
       [0.07, 0.65, 0.14, 0.14],
       [0.95, 0.01, 0.01, 0.03],
       [0.02, 0.93, 0.01, 0.04],
       [0.51, 0.02, 0.32, 0.14],
       [0.03, 0.86, 0.03, 0.08],
       [0.97, 0.  , 0.01, 0.02],
       [0.01, 0.94, 0.01, 0.04],
       [1.  , 0.  , 0.  , 0.  ],
       [0.98, 0.  , 0.01, 0.01],
       [0.1 , 0.11, 0.55, 0.24],
       [0.1 , 0.67, 0.06, 0.17],
       [0.02, 0.91, 0.03, 0.05],
       [0.01, 0.94, 0.01, 0.04],
       [0.11, 0.7 , 0.07, 0.12],
       [0.48, 0.24, 0.15, 0.13],
       [0.08, 0.73, 0.07, 0.12],
       [0.08, 0.71, 0.07, 0.14],
       [0.95, 0.01, 0.01, 0.03],
       [0.14, 0.63, 0.07, 0.16],
       [0.98, 0.  , 0.01, 0.01],
       [0.02, 0.92, 0.02, 0.05],
       [0.03, 0.92, 0.01, 0.04],
       [0.02, 0.93, 0.01, 0.04],
       [0.

In [24]:
y_test

array(['Depression', 'Depression', 'Depression', 'Depression',
       'Depression', 'Drugs', 'Depression', 'Depression', 'Alcohol',
       'Depression', 'Alcohol', 'Depression', 'Alcohol', 'Depression',
       'Alcohol', 'Alcohol', 'Drugs', 'Depression', 'Suicide',
       'Depression', 'Depression', 'Alcohol', 'Depression', 'Depression',
       'Alcohol', 'Suicide', 'Alcohol', 'Depression', 'Depression',
       'Depression', 'Depression', 'Depression', 'Depression',
       'Depression', 'Alcohol', 'Alcohol', 'Depression', 'Depression',
       'Depression', 'Depression', 'Alcohol', 'Depression', 'Depression',
       'Alcohol', 'Alcohol', 'Depression', 'Depression', 'Depression',
       'Depression', 'Depression', 'Depression', 'Depression', 'Alcohol',
       'Drugs', 'Depression', 'Alcohol', 'Depression', 'Depression',
       'Depression', 'Depression', 'Depression', 'Depression', 'Alcohol',
       'Depression', 'Alcohol', 'Depression', 'Depression', 'Depression',
       'Suicide', 'Alc

In [25]:
Sub= testing.iloc[:,1:500].values
preds = classifier.predict_proba(Sub)
preds = np.around(preds, decimals=3)

In [26]:
Submission = testing.iloc[:, 0].to_frame()
 # Depression,Alcohol,Suicide,Drugs
Submission['Depression']=preds[:,1]
Submission['Alcohol']=preds[:,0]
Submission['Drugs']=preds[:,2]
Submission['Suicide']=preds[:,3]
Submission.head(10)

,ID,Depression,Alcohol,Drugs,Suicide
0,02V56KMO,0.814,0.053,0.040,0.093
1,03BMGTOK,0.919,0.012,0.020,0.049
2,03LZVFM6,0.989,0.001,0.003,0.007
3,0EPULUM5,0.846,0.013,0.045,0.095
4,0GM4C5GD,0.102,0.237,0.490,0.171
5,0IYM8BYT,0.723,0.084,0.090,0.102
6,0JFJVD28,0.763,0.075,0.057,0.105
7,0OZGUFOF,0.884,0.024,0.030,0.062
8,0WP999ZX,0.000,0.994,0.002,0.004
9,0ZA45Y5F,0.189,0.601,0.083,0.127


In [27]:
test.head(10)

,ID,text
0,02V56KMO,How to overcome bad feelings and emotions
1,03BMGTOK,I feel like giving up in life
2,03LZVFM6,I was so depressed feel like got no strength t...
3,0EPULUM5,I feel so low especially since I had no one to...
4,0GM4C5GD,can i be successful when I am a drug addict?
5,0IYM8BYT,"I am dissapointed,I need a sense of closure"
6,0JFJVD28,I lost hope
7,0OZGUFOF,I feel unloved and uncared for
8,0WP999ZX,What are factors that leads to addiction of al...
9,0ZA45Y5F,"Is the advice of drinking moderatly ,prudent?"


In [28]:
Submission.to_csv('Submissions/Submission_v1.csv', index=False)